Questions :

1.1 Chargement des données

● Votre première tâche consiste à importer les bibliothèques Python nécessaires pour
réaliser ce TD, telles que Pandas et Numpy.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

● Ensuite, chargez le fichier de données correspondant (solar2021UNLV.csv) dans
un DataFrame nommé df_solar. Quelle colonne devez-vous utiliser comme index ?
Pourquoi ?

In [ ]:
df_solar = pd.read_csv('../dataset/solar2021UNLV.csv', index_col=0, parse_dates=True)
# La colonne d'index choisie est la première (probablement Date/Time) car elle permet de réaliser
# des analyses temporelles précises et de manipuler les données par tranches horaires.
df_solar.head()

1.2 Exploration des données

Dans cette partie, nous allons examiner le jeu de données afin de vous permettre de mieuxcomprendre les informations dont vous disposez.

1. Quelle est la taille du DataFrame chargé ?

In [ ]:
print(f"La taille du DataFrame est : {df_solar.shape}")

2. Affichez les 10 premières lignes et les 10 dernières lignes du DataFrame df_solar.

In [ ]:
print("10 premières lignes :")
display(df_solar.head(10))
print("\n10 dernières lignes :")
display(df_solar.tail(10))

3. Listez les noms des colonnes présentes dans ce jeu de données ainsi que leur type
de données.

In [ ]:
print("Informations sur les colonnes et types de données :")
df_solar.info()

4. Quelles variables (features) peuvent contenir des valeurs aberrantes (outliers) ?
Vous devez justifier votre réponse à l’aide d’un graphique, par exemple un boxplot.


In [ ]:
# Sélection de colonnes numériques pertinentes pour détecter les outliers
# On filtre pour ne garder que les colonnes numériques
numeriques = df_solar.select_dtypes(include=[np.number])
plt.figure(figsize=(15, 8))
sns.boxplot(data=numeriques.iloc[:, :10]) # On affiche les 10 premières colonnes numériques pour la lisibilité
plt.xticks(rotation=45)
plt.title("Boxplot des variables (10 premières)")
plt.show()
# Certaines variables comme GHI ou DNI peuvent présenter beaucoup d'outliers si les capteurs ont des pics de mesure
# ou si l'on compare des valeurs de nuit (0) avec des valeurs de plein soleil.

1.3 Manipulation des données

5. Comme vous avez pu le remarquer, le DataFrame df contient une ou plusieurs
colonnes inutiles. Supprimez la ou les colonnes non souhaitées.

In [ ]:
# Suppression des colonnes potentiellement inutiles (redondantes avec l'index temporel)
cols_to_drop = ['year', 'DOY', 'PST']
df_solar = df_solar.drop(columns=[c for c in cols_to_drop if c in df_solar.columns])
print("Colonnes restantes :", df_solar.columns.tolist())
df_solar.head()

6. Vérifiez la présence de valeurs manquantes (NaN) dans le DataFrame df_ref, puis
supprimez toutes les lignes contenant des valeurs manquantes.

In [ ]:
print(f"Nombre de valeurs manquantes par colonne :\n{df_solar.isnull().sum()}")
df_solar = df_solar.dropna()
print(f"\nTaille après suppression des NaNs : {df_solar.shape}")

7. Vérifiez la présence de doublons dans ce DataFrame, puis supprimez toutes les
lignes dupliquées.

In [ ]:
print(f"Nombre de doublons détectés : {df_solar.duplicated().sum()}")
df_solar = df_solar.drop_duplicates()
print(f"Taille après suppression des doublons : {df_solar.shape}")

1.4 Analyse des données et visualisation

8. Quelle est la valeur maximale de la variable “GHI” et à quelle date correspond-elle ?

In [ ]:
max_ghi = df_solar['ghi'].max()
date_max_ghi = df_solar['ghi'].idxmax()
print(f"La valeur maximale de GHI est : {max_ghi} W/m²")
print(f"Elle a été mesurée le : {date_max_ghi}")

9. Quelle est la valeur moyenne de “GHI” :

○ globale ?

In [ ]:
mean_ghi_global = df_solar['ghi'].mean()
print(f"Valeur moyenne globale de GHI : {mean_ghi_global:.2f} W/m²")

○ par mois ?

In [ ]:
mean_ghi_month = df_solar.groupby(df_solar.index.month)['ghi'].mean()
print("Moyenne de GHI par mois (1=Janvier, 12=Décembre) :")
print(mean_ghi_month)

○ par saison ?

In [ ]:
def get_season(month):
    if month in [12, 1, 2]: return 'Hiver'
    elif month in [3, 4, 5]: return 'Printemps'
    elif month in [6, 7, 8]: return 'Été'
    else: return 'Automne'

df_solar['season'] = df_solar.index.month.map(get_season)
mean_ghi_season = df_solar.groupby('season')['ghi'].mean()
print("Moyenne de GHI par saison :")
print(mean_ghi_season)

10. Tracez l’évolution de “GHI” pour une journée donnée.
Puis, tracez un second graphique pour cinq jours consécutifs à partir d’une date
choisie.

Vous pouvez extraire les valeurs de GHI pour une période donnée à partir de la
série temporelle en utilisant le slicing suivant :
df_ref['2021-06-01':'2021-06-02']['ghi']

In [ ]:
plt.figure(figsize=(15, 10))

plt.subplot(2, 1, 1)
df_solar['2021-06-21']['ghi'].plot(color='blue')
plt.title("Évolution du GHI sur une journée (21 Juin 2021)")
plt.ylabel("GHI (W/m²)")
plt.grid(True)

plt.subplot(2, 1, 2)
df_solar['2021-06-21':'2021-06-25']['ghi'].plot(color='orange')
plt.title("Évolution du GHI sur 5 jours consécutifs")
plt.ylabel("GHI (W/m²)")
plt.grid(True)

plt.tight_layout()
plt.show()

11. Tracez la variation de “GHI” par jour et par mois.
Pouvez-vous réaliser cela sans ajouter de nouvelles colonnes correspondant au jour
et au mois ? Si oui, expliquez comment

12. Tracez l’évolution de :

○ GHI

○ DHI

○ DNI

○ la température

en fonction de la date (jour), sous forme de sous-graphiques (subplots), un graphique par
variable.

13. Tracez la variation de la température en fonction du rayonnement solaire “GHI”.
Que pouvez-vous conclure sur la relation entre ces deux variables ?

14. Tracez la matrice de corrélation du jeu de données sous forme de carte
thermique (heatmap) afin de mieux comprendre les relations entre toutes les
variables du dataset.

● Utilisez la fonction corr() de Pandas pour calculer la matrice de corrélation.

● Stockez le résultat dans une variable appelée corr

Ces valeurs permettent d’analyser les relations entre les colonnes :

● Une corrélation parfaitement positive est égale à +1.

● Une corrélation parfaitement négative est égale à –1

Ensuite, répondez aux questions suivantes :

● En observant la diagonale principale (de gauche à droite) de la matrice de
corrélation, pourquoi cette diagonale est-elle remplie de valeurs égales à 1 ?
Expliquez.

● Toujours en observant la matrice de corrélation, vous remarquerez que les valeurs
sont symétriques : les valeurs situées sous la diagonale ont un équivalent au-dessus.
Pourquoi la matrice de corrélation est-elle symétrique ? Expliquez

● De nombreuses paires de variables présentent une corrélation proche de zéro.
Que signifie une corrélation proche de zéro ?

● Quelles sont les variables qui présentent les corrélations les plus fortes ?
Pouvez-vous les lister ?

15. Tracez un nouveau graphique de corrélation entre le rayonnement “GHI” et les
autres variables du dataset.

● Rappelez que la fonction corr() utilise par défaut la méthode de corrélation de
Pearson.

Répondez ensuite aux questions suivantes :

● Quelles variables présentent la plus forte corrélation avec “GHI” ?
Listez-les.

● Quelles variables ne présentent pas ou peu de corrélation avec “GHI” ?
Listez-les.

● Reproduisez la même analyse en utilisant d’autres méthodes de corrélation
disponibles dans Pandas :

○ le coefficient de corrélation de Kendall (Kendall Tau),

○ le coefficient de corrélation de Spearman.

● Quelles conclusions pouvez-vous tirer en comparant les résultats obtenus avec les
méthodes Pearson, Kendall et Spearman ?

16. Créez un nuage de points (scatter plot) pour illustrer la relation entre :

● l’angle d’azimut (Azimuth Angle) en abscisse,

● le rayonnement GHI en ordonnée.

Observez-vous un motif ou une structure particulière dans ce graphique ? Justifiez votre
réponse

17. Créez un nuage de points pour analyser la relation entre :

● l’angle zénithal (Zenith Angle [degrees]) en abscisse,

● le rayonnement GHI en ordonnée.

Existe-t-il une tendance ou un comportement remarquable dans ce graphique ? Expliquez.

18. Créez un nuage de points représentant :

● l’angle d’azimut en abscisse,

● l’angle zénithal en ordonnée,

et utilisez la variable “GHI” comme code couleur pour les points.

● Le résultat obtenu confirme-t-il les conclusions précédentes concernant la relation
entre GHI, azimut et zénith ?

● Justifiez votre réponse à l’aide du graphique.

19. Affichez un graphique de densité de probabilité bidimensionnelle (KDE 2D) pour
analyser :

● la relation entre le rayonnement GHI et la température.

Ensuite, réalisez un autre KDE 2D pour étudier la relation entre :

● la vitesse du vent (wind speed),

● la direction du vent (wind direction).


20. Tracez la distribution des variables suivantes sous forme de sous-graphiques
(subplots) :

● la vitesse du vent

● la direction du vent,

● la température

21. Tracez un graphique KDE bidimensionnel (2D) illustrant la distribution conjointe
des variables :

● DHI (Diffuse Horizontal Irradiance),

● DNI (Direct Normal Irradiance)

1.5 Investigation des différences

22. Créez une figure unique contenant les courbes KDE (qui montrent la densité)
de ‘DHI’ et ‘DNI’

23. Fournissez un pair plot qui représente toutes les relations par paires dans le
dataset en utilisant la méthode correspondante de la bibliothèque Seaborn.
Que pouvez-vous en conclure ?

1.6 Investigation supplémentaire
Créez un nuage de points 3D avec les variables (x, y, z) : où ‘DNI’, ‘DHI’ et ‘GHI’
représentent respectivement les axes x, y et z.
Vous pouvez utiliser le code suivant :

ax = plt.axes(projection='3d')

ax.scatter3D(x, y, z)

● Que pouvez-vous conclure à partir de ce graphique ?

1.7 Échantillonnage descendant et ascendant des séries temporelles
Lors de l’évaluation des ressources solaires, il se peut que vous ayez besoin d’une
résolution temporelle différente de celle de vos données pour une partie particulière de
l’analyse. Dans ces cas, il est possible de rééchantillonner à la baisse (down-sampling) ou à
la hausse (up-sampling) les données à différentes résolutions temporelles en utilisant deux
méthodes de la bibliothèque Pandas appelées resample et asfreq.
Selon vos besoins, vous choisirez l’une ou l’autre méthode.

Quelle que soit la méthode utilisée, les deux nécessitent un DataFrame avec un
DatetimeIndex, soit conscient du temps (time-aware / localisé), soit non conscient du temps
(time-naive / non localisé).

Quelle est la différence entre ces deux types ? Essayez d’expliquer la différence en détail

● Tracez la variation de ‘GHI’ pour des intervalles de 30 minutes, 1 heure et 1 semaine.
Que pouvez-vous en conclure ?

● Tracez la variation de ‘GHI’, ‘DHI’, ‘DNI’ et de la température pour des intervalles de
deux semaines sous forme de subplots.